# Hidden-item evaluation

Use validation for development and hold test results for final reporting. Exact-item recovery is an imperfect proxy for outfit compatibility. Results on the fixture are synthetic.


In [ ]:
import os, sys
from pathlib import Path
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ARCHITECTURE.md').is_file())
sys.path.insert(0, str(root))
from pipeline.prepare_dataset import read_json
fixture = root / 'data/processed/fixture'
manifest_path = Path(os.environ.get('STYLIST_MANIFEST', fixture / 'manifest.json'))
manifest = read_json(manifest_path)
image_root = Path(os.environ.get('STYLIST_IMAGE_ROOT', fixture))
print('Dataset:', manifest['dataset'], '| Revision:', manifest['revision'])

In [ ]:
from backend.app.recommendations.engine import Catalog, DEFAULT_POLICY
snapshot_env = os.environ.get('STYLIST_SNAPSHOT')
snapshot = Path(snapshot_env if snapshot_env else read_json(fixture / 'fixture.json')['snapshot'])
catalog = Catalog(snapshot)

from pipeline.run_evaluation import run
import matplotlib.pyplot as plt
split = os.environ.get('STYLIST_SPLIT', 'valid')
report = run(manifest_path, snapshot, DEFAULT_POLICY, split)
print('Method             Recall@3')
print('CLIP scores        ', report['clip']['recall_at_3'])
print('Random mean        ', report['random']['recall_at_3_mean'])
print('MRR@3:', report['clip']['mrr_at_3'])
print('Exclusions:', report['exclusion_reasons'])
misses = [case for case in report['cases'] if 'clip_rank' in case and case['clip_rank'] is None]
print('First five misses:', misses[:5])
if report['eligible_cases']:
    plt.bar(['Cosine retrieval', 'Random baseline'], [report['clip']['recall_at_3'], report['random']['recall_at_3_mean']])
    plt.ylim(0, 1)
    plt.ylabel('Recall@3')
    plt.show()